# Knowledge Graphs & LLMs

## What is a Knowledge Graph?
A Knowledge Graph (KG) stores information as a network of entities and relationships:

```
(Albert Einstein) --[born_in]--> (Germany)
(Albert Einstein) --[developed]--> (Theory of Relativity)
(Albert Einstein) --[worked_at]--> (Princeton University)
```

Each fact is a **triple**: (Subject, Predicate, Object) = (head, relation, tail)

## Knowledge Graph Embeddings

### TransE
Models relations as translations in embedding space:
$$h + r \approx t$$

Score: $f(h,r,t) = -\|h + r - t\|$

### RotatE
Models relations as rotations in complex space:
$$t = h \circ r \quad \text{where } |r_i| = 1$$

### TransR
Projects entities into relation-specific spaces:
$$h_r = M_r h, \quad t_r = M_r t$$
$$f(h,r,t) = -\|h_r + r - t_r\|^2$$

## GraphRAG (Microsoft)

GraphRAG builds a KG from documents, then uses it for retrieval:
1. **Chunk** documents
2. **Extract** entities and relationships using LLM
3. **Build** community hierarchy using Leiden algorithm
4. **Generate** community summaries
5. **Query**: global (community reports) or local (graph search)

In [1]:
# Building a knowledge graph with NetworkX
import networkx as nx
import matplotlib.pyplot as plt

# Create directed graph
G = nx.DiGraph()

# Add triples (subject, object, relation)
triples = [
    ("Albert Einstein", "Germany", "born_in"),
    ("Albert Einstein", "Theory of Relativity", "developed"),
    ("Albert Einstein", "Princeton", "worked_at"),
    ("Albert Einstein", "Nobel Prize Physics", "received"),
    ("Theory of Relativity", "Physics", "field_of"),
    ("Germany", "Europe", "located_in"),
    ("Princeton", "USA", "located_in"),
]

for h, t, r in triples:
    G.add_edge(h, t, relation=r)

# Query the graph
def query_entity(G, entity):
    outgoing = [(u, G[u][v]['relation'], v) for u, v in G.out_edges(entity)]
    incoming = [(u, G[u][v]['relation'], v) for u, v in G.in_edges(entity)]
    return outgoing + incoming

facts = query_entity(G, "Albert Einstein")
print("Facts about Albert Einstein:")
for h, r, t in facts:
    print(f"  ({h}) --[{r}]--> ({t})")

print(f"\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Facts about Albert Einstein:
  (Albert Einstein) --[born_in]--> (Germany)
  (Albert Einstein) --[developed]--> (Theory of Relativity)
  (Albert Einstein) --[worked_at]--> (Princeton)
  (Albert Einstein) --[received]--> (Nobel Prize Physics)

Graph: 8 nodes, 7 edges


In [2]:
# Extract KG from text using LLM
KG_EXTRACT_CODE = '''
from openai import OpenAI
import json

client = OpenAI()

def extract_triples(text):
    prompt = f"""Extract knowledge graph triples from this text.
Return JSON: {{"triples": [["subject", "relation", "object"], ...]}}

Text: {text}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)["triples"]

text = "Marie Curie was born in Warsaw, Poland. She won two Nobel Prizes, "
       "in Physics (1903) and Chemistry (1911). She discovered polonium and radium."

triples = extract_triples(text)
for triple in triples:
    print(f"({triple[0]}) --[{triple[1]}]--> ({triple[2]})")
'''
print(KG_EXTRACT_CODE)


from openai import OpenAI
import json

client = OpenAI()

def extract_triples(text):
    prompt = f"""Extract knowledge graph triples from this text.
Return JSON: {{"triples": [["subject", "relation", "object"], ...]}}

Text: {text}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)["triples"]

text = "Marie Curie was born in Warsaw, Poland. She won two Nobel Prizes, "
       "in Physics (1903) and Chemistry (1911). She discovered polonium and radium."

triples = extract_triples(text)
for triple in triples:
    print(f"({triple[0]}) --[{triple[1]}]--> ({triple[2]})")



In [3]:
# Neo4j Graph Database
NEO4J_CODE = '''
# pip install neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))

def create_triple(tx, h, r, t):
    query = f"""
    MERGE (h:Entity {{name: $h}})
    MERGE (t:Entity {{name: $t}})
    MERGE (h)-[:{r.upper().replace(' ', '_')} {{relation: $r}}]->(t)
    """
    tx.run(query, h=h, r=r, t=t)

def query_entity_neo4j(tx, entity):
    result = tx.run("""
    MATCH (e:Entity {name: $name})-[r]->(t)
    RETURN type(r) as relation, t.name as target
    """, name=entity)
    return [(row["relation"], row["target"]) for row in result]

# Insert triples
with driver.session() as session:
    for h, t, r in triples:
        session.execute_write(create_triple, h, r, t)

# Query
with driver.session() as session:
    facts = session.execute_read(query_entity_neo4j, "Albert Einstein")
    for rel, target in facts:
        print(f"  [{rel}] -> {target}")
'''
print(NEO4J_CODE)


# pip install neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))

def create_triple(tx, h, r, t):
    query = f"""
    MERGE (h:Entity {{name: $h}})
    MERGE (t:Entity {{name: $t}})
    MERGE (h)-[:{r.upper().replace(' ', '_')} {{relation: $r}}]->(t)
    """
    tx.run(query, h=h, r=r, t=t)

def query_entity_neo4j(tx, entity):
    result = tx.run("""
    MATCH (e:Entity {name: $name})-[r]->(t)
    RETURN type(r) as relation, t.name as target
    """, name=entity)
    return [(row["relation"], row["target"]) for row in result]

# Insert triples
with driver.session() as session:
    for h, t, r in triples:
        session.execute_write(create_triple, h, r, t)

# Query
with driver.session() as session:
    facts = session.execute_read(query_entity_neo4j, "Albert Einstein")
    for rel, target in facts:
        print(f"  [{rel}] -> {target}")



In [4]:
# GraphRAG with Microsoft's library
GRAPHRAG_CODE = '''
# pip install graphrag
# Initialize
# graphrag init --root ./my_project
# Place documents in ./my_project/input/
# graphrag index --root ./my_project
# graphrag query --root ./my_project --method global "What are the main themes?"
# graphrag query --root ./my_project --method local  "Who is Einstein?"

# Python API:
from graphrag.query.cli import run_global_search, run_local_search

# Global search: uses community summaries for broad questions
result = run_global_search("./my_project", "What are the main topics?", verbose=False)
print(result)

# Local search: uses entity embeddings for specific questions
result = run_local_search("./my_project", "Tell me about Einstein", verbose=False)
print(result)
'''
print(GRAPHRAG_CODE)


# pip install graphrag
# Initialize
# graphrag init --root ./my_project
# Place documents in ./my_project/input/
# graphrag index --root ./my_project
# graphrag query --root ./my_project --method global "What are the main themes?"
# graphrag query --root ./my_project --method local  "Who is Einstein?"

# Python API:
from graphrag.query.cli import run_global_search, run_local_search

# Global search: uses community summaries for broad questions
result = run_global_search("./my_project", "What are the main topics?", verbose=False)
print(result)

# Local search: uses entity embeddings for specific questions
result = run_local_search("./my_project", "Tell me about Einstein", verbose=False)
print(result)



## Additional Learning Resources

### Papers
- [GraphRAG](https://arxiv.org/abs/2404.16130) Edge et al., 2024
- [TransE](https://proceedings.neurips.cc/paper_files/paper/2013/file/1cecc7a77928ca8133fa24680a88d2f9-Paper.pdf)
- [RotatE](https://arxiv.org/abs/1902.10197)
- [KGRAG survey](https://arxiv.org/abs/2408.08921)

### Tools
- [Microsoft GraphRAG](https://github.com/microsoft/graphrag)
- [Neo4j](https://neo4j.com/docs/)
- [LangChain Graph](https://python.langchain.com/docs/how_to/graph_constructing/)
- [LlamaIndex KG Index](https://docs.llamaindex.ai/en/stable/examples/index_structs/knowledge_graph/)
- [PyKEEN](https://github.com/pykeen/pykeen) KG embedding library